# Example 1: Transfer-function visualization

This notebook reproduces the paper Example 1 using **synthetic data** that simulates
a second-order low-pass system with known transfer function.

It demonstrates:
- Computing a transfer function from two `TimeSeries` using `TimeSeries.transfer_function()`
- Converting a complex `FrequencySeries` to dB magnitude via `to_db()`
- Extracting phase in degrees via `degree()`
- Producing a Bode-style two-panel plot via `gwexpy.plot.Plot`

> **Note on DTT XML**: In real workflows the `FrequencySeriesMatrix` would be read
> directly from a DTT XML file: `FrequencySeriesMatrix.read("file.xml", format="dttxml", products="TF")`.
> Here we construct an equivalent `FrequencySeries` from synthetic time-domain data.

In [ ]:
import matplotlib

matplotlib.use('Agg')

import numpy as np
from scipy import signal as scipy_signal

from gwexpy.timeseries import TimeSeries  # gwexpy extension with mode='steady' support

## 1. Generate synthetic input/output time series

We create a known second-order low-pass filter (cutoff 50 Hz, Q=0.707) and filter
a white-noise input to get the output.  The estimated TF should recover the known
magnitude and phase response.

In [ ]:
rng = np.random.default_rng(42)
sr = 2048          # sample rate [Hz]
duration = 60      # seconds
n = sr * duration

# White-noise excitation
x = rng.standard_normal(n)

# Second-order low-pass: cutoff 50 Hz, Q = 0.707 (Butterworth)
f_cut = 50.0  # Hz
sos = scipy_signal.butter(2, f_cut / (sr / 2), btype='low', output='sos')
y = scipy_signal.sosfilt(sos, x)

ts_in  = TimeSeries(x, t0=0, sample_rate=sr, name='INPUT')
ts_out = TimeSeries(y, t0=0, sample_rate=sr, name='OUTPUT')

print(f'Sample rate : {sr} Hz')
print(f'Duration    : {duration} s')
print(f'Cutoff freq : {f_cut} Hz')

## 2. Estimate transfer function

`TimeSeries.transfer_function(other, mode='steady')` computes H(f) = CSD(in,out) / PSD(in)
using the Welch estimator and returns a `FrequencySeries`.

In [ ]:
tf = ts_in.transfer_function(ts_out, fftlength=4.0, overlap=2.0, mode='steady')

print(f'FrequencySeries length : {len(tf)}')
print(f'Frequency resolution   : {tf.df:.3f} Hz')
print(f'Max frequency          : {tf.frequencies.value[-1]:.1f} Hz')

## 3. Bode-style plot via gwexpy high-level API

This mirrors the paper Listing 2 workflow:
- `to_db()` converts the complex TF to dB magnitude
- `degree()` extracts the phase in degrees
- `Plot(..., separate=True, sharex=True)` renders both as stacked subplots

In [ ]:
from gwexpy.plot import Plot

mag_db = tf.to_db()
phase_deg = tf.degree()

plot = Plot(mag_db, phase_deg, separate=True, sharex=True)
plot.show()

## 4. Validation: compare estimated TF to the known analytical response

In [ ]:
import matplotlib.pyplot as plt

freqs = tf.frequencies.value
# Analytical magnitude of the 2nd-order LP filter
w, H_theory = scipy_signal.sosfreqz(sos, worN=freqs, fs=sr)
H_theory_db = 20 * np.log10(np.abs(H_theory) + 1e-30)

est_db = mag_db.value

# Compare in the 1–200 Hz band where SNR is good
mask = (freqs >= 1.0) & (freqs <= 200.0)
rms_error_db = np.sqrt(np.mean((est_db[mask] - H_theory_db[mask])**2))
print(f'RMS error between estimated and theoretical TF magnitude: {rms_error_db:.2f} dB')

# Should be within a few dB given 60 s of data
assert rms_error_db < 3.0, (
    f'TF estimation error too large: {rms_error_db:.2f} dB (expected < 3 dB)'
)
print('Validation passed.')
plt.close('all')